# Grid as a Weighted Directed Network

Bidding zones are nodes, cross-border interconnectors are edges, and physical flow is the edge weight &ndash; the grid as a network rather than a collection of individual border leaderboards.

This article asks three questions: which zones are structurally important once you account for *all* their connections, not just their single busiest border (centrality); do the clusters that physical flow naturally forms line up with the market-coupling regions that actually clear prices (community detection vs CCR mechanism); and how much of the network's integrity rests on a handful of zones or borders (robustness).

In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import networkx as nx
import pandas as pd
import plotly.express as px
from common import (
    CCR_MECHANISM_COLORS,
    ZONE_REGIONS,
    border_betweenness,
    border_ccr_mechanism,
    border_robustness_curve,
    build_flow_graph,
    build_rent_graph,
    edge_robustness_curve,
    edge_unit_of_failure_chart,
    fetch_flows_range,
    network_graph_chart,
    node_robustness_chart,
    node_robustness_curve,
    save_chart,
)

## Pulling a year of flow data

Same window as the self-organized-criticality article: a full year ending 10 June 2026, the day Core's Advanced Hybrid Coupling (AHC) went live. Keeping the whole sample inside one stable market-structure regime matters more here than for the SOC piece &ndash; a network built from flows that straddle a capacity-allocation reform would mix two different topologies into one snapshot.

In [ ]:
date_from = "2025-06-10"
date_to = "2026-06-10"

flows = fetch_flows_range(date_from, date_to)
print(f"{len(flows):,} observations, {flows['border'].nunique()} borders, {date_from} to {date_to}")
flows.head()

## Building the flow network

Each border becomes up to two directed edges, not one: `net_flow_mw` is signed (positive one way, negative the other), and the `bidding_zone_from`/`bidding_zone_to` columns are a fixed nominal pair per border, not the actual flow direction. Summing the positive and negative parts separately gives the real directed throughput each way &ndash; collapsing on the nominal column direction instead would silently throw away every border that mostly flows "backward," which turned out to zero out *every* cycle in the network during scoping.

UA_IPS borders are dropped, same convention as the congestion and SOC articles: wartime grid conditions and emergency EU synchronisation drive those numbers, not ordinary market behaviour.

In [ ]:
G = build_flow_graph(flows)
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"strongly connected: {nx.is_strongly_connected(G)}")

## Centrality

Three measures, three different questions:

- **PageRank** (weighted, directed) &ndash; which zones are structurally important once you weight by *who* flows into them, not just how much. Preferred over plain eigenvector centrality here: eigenvector centrality can degenerate to all-zero on a directed graph that isn't strongly connected, and while this window's graph checks out as strongly connected, PageRank is the robust default that doesn't depend on that holding for every future window.
- **In/out-strength** (weighted degree) &ndash; the simple baseline, total MW flowing in and out of each zone.
- **Betweenness** (weighted by inverse flow, since betweenness needs distance/cost semantics and raw flow is a strength, not a cost) &ndash; which zones sit on the most paths between other zone pairs, i.e. transit hubs rather than high-volume endpoints.

In [ ]:
pagerank = nx.pagerank(G, weight="weight")

G_dist = G.copy()
for _, _, d in G_dist.edges(data=True):
    d["distance"] = 1 / d["weight"]
betweenness = nx.betweenness_centrality(G_dist, weight="distance")

centrality = pd.DataFrame(
    {
        "pagerank": pagerank,
        "betweenness": betweenness,
        "strength_in_mw": dict(G.in_degree(weight="weight")),
        "strength_out_mw": dict(G.out_degree(weight="weight")),
    }
).sort_values("pagerank", ascending=False)
centrality.head(10)

In [ ]:
fig = px.bar(
    centrality.reset_index(names="zone").sort_values("pagerank").tail(10),
    x="pagerank",
    y="zone",
    orientation="h",
    title=f"Top 10 zones by PageRank, {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "pagerank_top10")

In [ ]:
fig = px.bar(
    centrality.reset_index(names="zone").sort_values("betweenness").tail(10),
    x="betweenness",
    y="zone",
    orientation="h",
    title=f"Top 10 zones by betweenness centrality (transit hubs), {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "betweenness_top10")

## Community detection vs market-coupling regions

Louvain community detection on the flow-weighted graph (direction collapsed, both-way weights summed &ndash; Louvain's modularity optimization is defined on undirected graphs) finds clusters from physical flow alone, with no knowledge of market structure. Comparing that against `border_ccr_mechanism()` &ndash; the real CCR capacity-allocation mechanism per border (Core flow-based, Nordic/Hansa/Baltic/SEE/SWE NTC, Switzerland explicit) &ndash; checks whether physical clustering actually lines up with how prices get coupled, or whether the two cut across each other.

Each zone gets a single CCR label: the mechanism of its highest-weighted incident border, since a zone can straddle more than one CCR.

In [ ]:
G_undirected = nx.Graph()
for u, v, d in G.edges(data=True):
    w = d["weight"]
    if G_undirected.has_edge(u, v):
        G_undirected[u][v]["weight"] += w
    else:
        G_undirected.add_edge(u, v, weight=w)

communities = nx.algorithms.community.louvain_communities(G_undirected, weight="weight", seed=42)
zone_community = {zone: f"community {i}" for i, comm in enumerate(communities) for zone in comm}
print(f"{len(communities)} communities: " + ", ".join(f"{{{', '.join(sorted(c))}}}" for c in communities))

In [ ]:
zone_mechanism_weight: dict[str, dict[str, float]] = {}
for u, v, d in G.edges(data=True):
    mechanism = border_ccr_mechanism(f"{u} -> {v}")
    for zone in (u, v):
        zone_mechanism_weight.setdefault(zone, {})
        zone_mechanism_weight[zone][mechanism] = zone_mechanism_weight[zone].get(mechanism, 0) + d["weight"]

zone_mechanism = {zone: max(weights, key=weights.get) for zone, weights in zone_mechanism_weight.items()}

comparison = pd.DataFrame(
    {
        "zone": list(G.nodes()),
        "community": [zone_community[z] for z in G.nodes()],
        "mechanism": [zone_mechanism[z] for z in G.nodes()],
    }
)
crosstab = pd.crosstab(comparison["community"], comparison["mechanism"])
purity = crosstab.max(axis=1).sum() / crosstab.sum().sum()
print(
    f"community-mechanism purity: {purity:.1%} (zones whose community's dominant mechanism matches their own)"
)
crosstab

In [ ]:
fig = network_graph_chart(
    G,
    groups=zone_community,
    color_map=None,
    title=f"Flow network, colored by Louvain community, {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "network_by_community")

In [ ]:
fig = network_graph_chart(
    G,
    groups=zone_mechanism,
    color_map=CCR_MECHANISM_COLORS,
    title=f"Flow network, colored by CCR capacity-allocation mechanism, {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "network_by_ccr_mechanism")

For most zones the two colorings above roughly agree &ndash; physical flow clusters line up with real CCR mechanism. But for interconnector-heavy small zones, flow diverges from both mechanism and geography in the same direction &ndash; one or two dominant physical links pull them into a different cluster than either label assigns (`NO_2`/`DK_1` pulled into the Core cluster via NordLink/DK interconnectors, `CH` pulled toward France/Iberia, the Baltics pulled toward Scandinavia). See the mismatch breakdown below.

## Robustness: random failure vs targeted attack

Removing nodes (zones) or edges (borders) one at a time and tracking how much of the network stays connected &ndash; measured as the largest weakly-connected component's share of all zones &ndash; separates two different failure modes. Removing in **PageRank/betweenness rank order** simulates a targeted attack or a worst-case cascading failure starting at the most important zone or border; removing in **random order**, averaged over many trials, simulates ordinary uncorrelated outages. The gap between the two curves, not either curve alone, is the finding: real networks are typically far more resilient to random failure than to targeted attack, and the size of that gap is a measure of how concentrated the network's integrity is in a few zones/borders.

In [ ]:
import random

n_random_trials = 50
targeted_order = list(centrality.sort_values("pagerank", ascending=False).index)

targeted = node_robustness_curve(G, targeted_order)
random_trials = [
    node_robustness_curve(G, random.sample(list(G.nodes()), G.number_of_nodes()))
    for _ in range(n_random_trials)
]
random_mean = pd.DataFrame(random_trials).mean().tolist()

node_robustness = pd.DataFrame(
    {
        "zones_removed": range(len(targeted)),
        "targeted (by PageRank)": targeted,
        "random (mean of 50)": random_mean,
    }
).set_index("zones_removed")

In [ ]:
def halfway_point(curve: list[float]) -> int:
    return next((i for i, v in enumerate(curve) if v < 0.5), len(curve))


n_zones = G.number_of_nodes()
t_zones, r_zones = halfway_point(targeted), halfway_point(random_mean)
trials = n_random_trials
print(f"targeted: giant component drops below 50% after removing {t_zones} of {n_zones} bidding zones")
print(
    f"random:   giant component drops below 50% after removing {r_zones} of {n_zones} bidding zones "
    f"(mean of {trials} trials)"
)

In [ ]:
fig = node_robustness_chart(node_robustness, date_from, date_to)
fig.show()
save_chart(fig, "robustness_node")

**Node removal** shows a plateau - the giant component holds roughly flat for both targeted and random removal before either one declines sharply, rather than the textbook pattern of targeted and random diverging from the first node removed.

In [ ]:
edge_betweenness = nx.edge_betweenness_centrality(G_dist, weight="distance")
targeted_edge_order = sorted(edge_betweenness, key=edge_betweenness.get, reverse=True)

targeted_edges = edge_robustness_curve(G, targeted_edge_order)
random_edge_trials = [
    edge_robustness_curve(G, random.sample(list(G.edges()), G.number_of_edges()))
    for _ in range(n_random_trials)
]
random_edge_mean = pd.DataFrame(random_edge_trials).mean().tolist()

edge_robustness = pd.DataFrame(
    {
        "borders_removed": range(len(targeted_edges)),
        "targeted (by betweenness)": targeted_edges,
        "random (mean of 50)": random_edge_mean,
    }
).set_index("borders_removed")

In [ ]:
n_borders = G.number_of_edges()
t_edges, r_edges = halfway_point(targeted_edges), halfway_point(random_edge_mean)
trials = n_random_trials
print(f"targeted: giant component drops below 50% after removing {t_edges} of {n_borders} borders")
print(
    f"random:   giant component drops below 50% after removing {r_edges} of {n_borders} borders "
    f"(mean of {trials} trials)"
)

The border count above (157) is misleading. `build_flow_graph` models a bidirectional border as two separate directed edges (one per direction), so removing "edges" removes each direction as an independent event at its own rank in the curve &ndash; a real outage takes out the whole physical interconnector, both directions, at once. That understates each removal's impact and inflates the apparent number of failures needed to collapse the network, since one physical border can count as two removal events. 78 of this network's 79 border pairs carry flow in both directions, so this isn't a corner case. The curve and halfway numbers above are kept for reference; the corrected version below removes by border (`frozenset({u, v})`, both directions atomically) instead.

In [ ]:
border_scores = border_betweenness(edge_betweenness)
border_order = sorted(border_scores, key=border_scores.get, reverse=True)

targeted_borders = border_robustness_curve(G, border_order)
random_border_trials = [
    border_robustness_curve(G, random.sample(border_order, len(border_order))) for _ in range(n_random_trials)
]
random_border_mean = pd.DataFrame(random_border_trials).mean().tolist()

border_robustness = pd.DataFrame(
    {
        "borders_removed": range(len(targeted_borders)),
        "targeted (by betweenness)": targeted_borders,
        "random (mean of 50)": random_border_mean,
    }
).set_index("borders_removed")

In [ ]:
fig = edge_unit_of_failure_chart(edge_robustness, border_robustness, date_from, date_to)
fig.show()
save_chart(fig, "robustness_edge_unit_of_failure")

TODO - does this topologically prove that N-1 works for ENTSOE? 

In [ ]:
n_border_pairs = len(border_order)
t_borders, r_borders = halfway_point(targeted_borders), halfway_point(random_border_mean)
print(f"targeted: giant component drops below 50% after removing {t_borders} of {n_border_pairs} borders")
print(
    f"random:   giant component drops below 50% after removing {r_borders} of {n_border_pairs} borders "
    f"(mean of {n_random_trials} trials)"
)

With borders as the unit of failure, targeted removal's halfway point moves from 120 of 157 directed edges to 48 of 79 border pairs (60.8% removed, vs. 76.4% under the old directed-edge count), and random removal's moves from 117 of 157 to 45 of 79 (57.0%, vs. 74.5%). The gap between targeted and random narrows in absolute terms (3 borders vs. the old 3-edge gap) but the qualitative story holds either way: targeted and random stay close through most of the collapse region, with targeted still edging ahead late, once the network is already fragmented. What changes is any number quoted as "X of Y borders": those should come from this corrected curve, not the directed-edge one above.

**Edge removal** mostly follows the classic story in both panels above: from roughly the first third of removals through the halfway point, targeted removal (by betweenness) leaves a *smaller* giant component than random removal of the same size - real disruption, with targeted and random halfway points landing close together (no meaningful redundancy bonus over the range that matters for an outage). Only past the collapse point, once the network is already fragmented, does targeted sit slightly above random for a while - the skeleton left by betweenness-first removal hangs together a bit better than a random remnant of the same size, consistent with most borders having multiple parallel interconnectors (N-1 design), so late removals increasingly hit redundant rather than unique links. A real but secondary effect, not a reason to expect the grid to shrug off a targeted attack on its most-central borders.

## How solid is that 65.1%?

Two checks before trusting the purity number: does the 6-community partition survive reasonable changes to how aggressively Louvain splits communities, and is 65.1% actually better than what a random graph with the same degree sequence would produce by chance.

In [ ]:
import numpy as np

resolution_sweep = []
for resolution in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    comms = nx.algorithms.community.louvain_communities(
        G_undirected, weight="weight", resolution=resolution, seed=42
    )
    comm_of = {zone: i for i, c in enumerate(comms) for zone in c}
    ct = pd.crosstab(
        pd.Series([comm_of[z] for z in G.nodes()], name="community"),
        pd.Series([zone_mechanism[z] for z in G.nodes()], name="mechanism"),
    )
    resolution_sweep.append(
        {
            "resolution": resolution,
            "n_communities": len(comms),
            "purity": ct.max(axis=1).sum() / ct.sum().sum(),
        }
    )

pd.DataFrame(resolution_sweep).set_index("resolution")

Community count and purity move together smoothly across the sweep, with no cliff-edge at the default `resolution=1.0` &ndash; the 6-community partition isn't an arbitrary artifact of the default setting.

In [ ]:
def rewired_purity(G_undirected: nx.Graph, zone_mechanism: dict[str, str], rng: random.Random) -> float:
    rewired = nx.double_edge_swap(
        G_undirected.copy(), nswap=G_undirected.number_of_edges() * 5, max_tries=10_000, seed=rng
    )
    comms = nx.algorithms.community.louvain_communities(rewired, weight="weight", seed=42)
    comm_of = {zone: i for i, c in enumerate(comms) for zone in c}
    ct = pd.crosstab(
        pd.Series([comm_of[z] for z in G.nodes()], name="community"),
        pd.Series([zone_mechanism[z] for z in G.nodes()], name="mechanism"),
    )
    return ct.max(axis=1).sum() / ct.sum().sum()


n_null_trials = 200
rng = random.Random(42)
null_purities = [rewired_purity(G_undirected, zone_mechanism, rng) for _ in range(n_null_trials)]
percentile = (np.array(null_purities) < purity).mean()
print(
    f"real purity {purity:.1%} vs null mean {np.mean(null_purities):.1%} "
    f"({percentile:.0%} of {n_null_trials} degree-preserving rewirings scored lower)"
)

In [ ]:
fig = px.histogram(
    pd.DataFrame({"purity": null_purities}),
    x="purity",
    nbins=20,
    title=f"Community-mechanism purity under degree-preserving rewiring ({n_null_trials} trials)",
)
fig.add_vline(x=purity, line_color="#8a2e2e", line_width=2, annotation_text=f"real network: {purity:.1%}")
fig.show()
save_chart(fig, "purity_null_model")

## Flow clusters vs. geography

`border_ccr_mechanism()` is the real market-coupling rule, but the article's earlier SOC and congestion work also leans on `ZONE_REGIONS` &ndash; a geographic approximation of the same CCR map. Checking both against the Louvain communities at once separates two different mismatches: zones where flow follows geography but not the market rule, and zones where flow doesn't follow either.

In [ ]:
zone_region = {zone: ZONE_REGIONS[zone] for zone in G.nodes()}

region_crosstab = pd.crosstab(comparison["community"], comparison["zone"].map(zone_region))
region_purity = region_crosstab.max(axis=1).sum() / region_crosstab.sum().sum()
print(f"community-region purity: {region_purity:.1%} (vs community-mechanism purity: {purity:.1%})")

mismatches = comparison[
    comparison.apply(
        lambda row: zone_mechanism[row["zone"]] != crosstab.loc[row["community"]].idxmax(),
        axis=1,
    )
].copy()
mismatches["region"] = mismatches["zone"].map(zone_region)
mismatches["community_dominant_region"] = mismatches["community"].map(
    lambda c: region_crosstab.loc[c].idxmax()
)
mismatches["verdict"] = mismatches.apply(
    lambda row: "geography wins" if row["region"] == row["community_dominant_region"] else "neither",
    axis=1,
)
mismatches[["zone", "community", "mechanism", "region", "verdict"]]

## Weighting by economic stakes instead of throughput

Every result so far weights edges by raw MW &ndash; a border is 'important' if a lot of power crosses it, regardless of price. Congestion rent (`value_eur`, already computed for the congestion-analysis article as `price_delta &times; net_flow_mw &times; duration`) measures something different: how much economic value that flow actually captured. A border can carry huge volume at a near-zero price gap (busy but economically unremarkable), or modest volume at a persistent price gap (quiet but high-stakes). Rebuilding the same directed graph with rent instead of MW, then comparing centrality rankings, separates *topologically* central zones from *economically* central ones &ndash; and the delta between the two rankings is the actual finding, not either ranking alone.

In [ ]:
G_rent = build_rent_graph(flows)
print(f"{G_rent.number_of_nodes()} nodes, {G_rent.number_of_edges()} edges")
print(f"strongly connected: {nx.is_strongly_connected(G_rent)}")

Same direction convention as the flow graph &ndash; each border's rent is attributed to whichever way the physical flow actually ran that hour (by the sign of `net_flow_mw`), so `G_rent` has the exact same node/edge shape as `G` and the two centrality rankings are directly comparable, zone for zone.

In [ ]:
pagerank_rent = nx.pagerank(G_rent, weight="weight")

G_rent_dist = G_rent.copy()
for _, _, d in G_rent_dist.edges(data=True):
    d["distance"] = 1 / d["weight"]
betweenness_rent = nx.betweenness_centrality(G_rent_dist, weight="distance")

rent_rank = pd.Series(pagerank_rent).rank(ascending=False)
flow_rank = centrality["pagerank"].rank(ascending=False)

rank_delta = pd.DataFrame(
    {
        "pagerank_rank_by_flow": flow_rank,
        "pagerank_rank_by_rent": rent_rank,
        "betweenness_rank_by_flow": centrality["betweenness"].rank(ascending=False),
        "betweenness_rank_by_rent": pd.Series(betweenness_rent).rank(ascending=False),
    }
)
rank_delta["pagerank_rank_shift"] = rank_delta["pagerank_rank_by_flow"] - rank_delta["pagerank_rank_by_rent"]
rank_delta["betweenness_rank_shift"] = (
    rank_delta["betweenness_rank_by_flow"] - rank_delta["betweenness_rank_by_rent"]
)
rank_delta.sort_values("pagerank_rank_shift", ascending=False)

In [ ]:
fig = px.bar(
    rank_delta.reset_index(names="zone")
    .reindex(rank_delta["pagerank_rank_shift"].abs().sort_values().index)
    .tail(10),
    x="pagerank_rank_shift",
    y="zone",
    orientation="h",
    title=(
        "Biggest PageRank rank shifts, flow-weighted vs rent-weighted "
        "(positive = more economically than topologically central)"
    ),
)
fig.show()
save_chart(fig, "pagerank_rank_shift")

## What this tells us about Europe's grid

**Centrality splits into two different kinds of "important."** PageRank ranks `IT_CNOR`, `IT_CSUD`, `HU`, `RO`, `IT_NORD` highest &ndash; these are high-*volume* zones, mostly Italy's internal zone-to-zone flows (Italy is split into 7 bidding zones, so a lot of its domestic transfer shows up as "cross-border" flow in this dataset). Betweenness tells a different story: `DE_LU` (0.63), `HU` (0.45), and `AT` (0.40) top that ranking &ndash; these are *transit* zones, sitting on the paths between other zone pairs rather than being high-volume endpoints themselves. Germany-Luxembourg is the clearest case: unremarkable PageRank, by far the highest betweenness &ndash; it's Europe's crossroads, not its biggest mover.

**Robustness shows almost no gap between targeted and random failure.** Removing zones by PageRank rank drops the giant component below 50% after 13 removals; random removal (averaged over 50 trials) takes 14. Removing borders by betweenness takes 120; random removal takes 117. Both pairs are close enough to call "no meaningful asymmetry" at this granularity &ndash; a departure from the classic scale-free "robust to random failure, fragile to targeted attack" signature, and worth taking at face value rather than forcing the expected story: at the continent-wide bidding-zone level, this network's integrity doesn't appear concentrated in a small set of critical nodes or borders the way a hub-and-spoke or scale-free topology would predict. That's consistent with the centrality split above &ndash; Europe's grid has distinct volume hubs (Italy's internal zones) and distinct transit hubs (`DE_LU`, `HU`, `AT`), but no single zone or border that is simultaneously both, which is likely why no small targeted set breaks the network dramatically faster than chance.

**Louvain communities partly line up with real market coupling, and the 65.1% number holds up.** 6 communities emerge from physical flow alone, matching geography reasonably well (a Balkans/SEE cluster, a Nordic-Baltic cluster, a southern-Italy-and-islands cluster, a Norway-only cluster, and two mixed Western/Central clusters). Purity against the real CCR mechanism (`border_ccr_mechanism`) is **65.1%** &ndash; and that number is solid on both checks it needed to pass: the partition is stable across a resolution sweep from 0.5 to 2.0 (4&ndash;9 communities, purity never drops below 65%, no cliff-edge at the default), and it's nowhere near a coincidence of this graph's degree sequence &ndash; 200 degree-preserving rewirings of the same graph average **39.4%** purity, with *none* scoring as high as the real network's 65.1%. Physical flow really does carry structure that tracks market-coupling mechanism far more than chance alone would predict.

**But the mismatching third is where this gets interesting, and it's not really about geography.** 15 of 43 zones land in a community whose dominant mechanism isn't their own. Checking those 15 against `ZONE_REGIONS` (a geographic approximation of the same CCR map) splits them roughly in half: 7 zones mismatch on mechanism but still agree with their community on *region* (e.g. `AL`/`GR`/`BG`/`MK`/`RS` &ndash; Balkans zones whose mechanism label is simply "Other, not in the CCR breakdown," so geography is doing fine here, the mechanism data just doesn't cover them). The other 8 agree with *neither* &ndash; and these are the real finding. `NO_2` and `DK_1` are both nominally Nordic-mechanism zones, but physical flow pulls them structurally into the same cluster as `AT`/`CZ`/`DE_LU`/`NL`/`PL` (community 1) &ndash; a Core-dominated group by both mechanism *and* region. That's not "flow approximately follows geography instead of market rules" &ndash; it's flow cutting across both labels at once, almost certainly because `NO_2`'s NordLink and `DK_1`'s interconnectors physically tie them into German/Dutch trading patterns more than into the rest of the Nordic synchronous area. Switzerland (`CH`) shows the same pattern from the other side: its own region label is Core, but its community is dominated by France/Iberia. The Baltic states (`EE`, `LV`, `LT`) land with Scandinavia rather than with each other, consistent with their subsea interconnectors to the Nordic grid mattering more, flow-wise, than their shared Baltic mechanism label or their own regional grouping.

The pattern across all 8 "neither" zones is the same: each is a small zone with one or two dominant interconnectors to a *different* region than the one its own mechanism/region label assigns it to, and physical flow follows those interconnectors, not the label. That's a genuine caution for using `ZONE_REGIONS` as a stand-in for real market structure in future ENTSO-E analysis &ndash; it's a reasonable approximation for the *majority* of zones, but for interconnector-heavy border zones specifically, geography and flow topology diverge from each other almost as often as they agree, and neither one should be assumed to track the other without checking.

**Weighting by economic stakes instead of MW shifts a specific handful of zones, cleanly.** Rebuilding the same directed graph with congestion rent (`value_eur`) instead of raw flow and recomputing PageRank/betweenness: `IT_SICI` and `IT_CALA` (Sicily, Calabria) jump 15 PageRank ranks higher under rent-weighting &ndash; modest absolute MW, but flows there capture an outsized price gap, i.e. genuinely scarce, economically valuable capacity rather than just throughput. `BG` and `ES` move the opposite way (13&ndash;14 ranks lower) &ndash; high-MW borders that, over this window, cleared at a comparatively small price differential: mechanically busy, economically unremarkable. The betweenness shift tells the equivalent transit-role story: `RO`, `IT_CNOR`, and `SI` lose 19&ndash;28 ranks (physically on many shortest paths, but those paths rarely carry much price-gap value), while `ME`, `NO_4`, and `RS` gain 18&ndash;27 (quieter in MW, but the specific connections they provide capture outsized economic stakes). `DE_LU`, `AT`, and `HU` barely move in either ranking &ndash; the zones already identified as structurally important are important for both reasons at once, which is itself informative: topological and economic centrality mostly agree at the top of the ranking and diverge in the middle, among zones that are flow-significant without being price-significant or vice versa.